# Pre-processing Raw ADNI Dataset
- Convert .nii format MRI images to png

In [12]:
#import libraries
import os
import nibabel as nib
import numpy as np
from PIL import Image
import shutil

In [5]:
# Define the input and output directories
input_dir = "../ADNI"  # Folder containing .nii or .nii.gz files
output_dir = "../2D_AXIAL"  # Folder where slices will be saved
classes = ["AD", "CN", "MCI"] # Categories

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Loop through all NIfTI files in the input directory
for cat in classes:
    input_dir_cat = input_dir + "/" + cat
    output_dir_cat = output_dir + "/" + cat
    os.makedirs(output_dir_cat, exist_ok=True)
    for nii_file in os.listdir(input_dir_cat):
        if nii_file.endswith(".nii") or nii_file.endswith(".nii.gz"):  # Process only NIfTI files
            nii_path = os.path.join(input_dir_cat, nii_file)

            try:
                # Load the NIfTI image
                nii_img = nib.load(nii_path)
                img_data = nii_img.get_fdata()
        
                # Normalize intensity values for better contrast (optional)
                img_data = (img_data - img_data.min()) / (img_data.max() - img_data.min()) * 255
                img_data = img_data.astype(np.uint8)
        
                # Create a subfolder for this file's slices
                subfolder = os.path.join(output_dir_cat, os.path.splitext(nii_file)[0])
                os.makedirs(subfolder, exist_ok=True)
        
                # Save axial slices
                for i in range(img_data.shape[2]):  # Loop through axial slices (along the 3rd axis)
                    slice_img = img_data[:, :, i]  # Extract the ith axial slice
                    slice_path = os.path.join(subfolder, f"slice_{i:03d}.png")
                    Image.fromarray(slice_img).save(slice_path)
        
                print(f"Processed {nii_file} and saved slices in {subfolder}")
                
            except Exception as e:
                print(f"Error processing {nii_file}: {e}. Skipping this file.")

print("All files have been processed!")

Processed I118924.nii and saved slices in ../2D_AXIAL/AD/I118924
Processed I64862.nii and saved slices in ../2D_AXIAL/AD/I64862
Processed I79577.nii and saved slices in ../2D_AXIAL/AD/I79577
Processed I89653.nii and saved slices in ../2D_AXIAL/AD/I89653
Processed I65374.nii and saved slices in ../2D_AXIAL/AD/I65374
Processed I122703.nii and saved slices in ../2D_AXIAL/AD/I122703
Processed I97069.nii and saved slices in ../2D_AXIAL/AD/I97069
Processed I38887.nii and saved slices in ../2D_AXIAL/AD/I38887
Processed I72805.nii and saved slices in ../2D_AXIAL/AD/I72805
Processed I106507.nii and saved slices in ../2D_AXIAL/AD/I106507
Processed I31326.nii and saved slices in ../2D_AXIAL/AD/I31326
Processed I83806.nii and saved slices in ../2D_AXIAL/AD/I83806
Processed I59677.nii and saved slices in ../2D_AXIAL/AD/I59677
Processed I118881.nii and saved slices in ../2D_AXIAL/AD/I118881
Processed I118880.nii and saved slices in ../2D_AXIAL/AD/I118880
Processed I82700.nii and saved slices in ../2

In [7]:
for cat in classes:
    output_dir_cat = output_dir + "/" + cat
    num_subfolders = 0
    total_files = 0

    # Iterate through the main folder
    for entry in os.scandir(output_dir_cat):
        if entry.is_dir():  # Check if it's a subfolder
            num_subfolders += 1
            # Count files in this subfolder
            total_files += sum(1 for _ in os.scandir(entry.path) if _.is_file())

    print(f"Number of {cat} subfolders: {num_subfolders}")
    print(f"Total number of files across all {cat} subfolders: {total_files}")

Number of AD subfolders: 80
Total number of files across all AD subfolders: 13154
Number of CN subfolders: 136
Total number of files across all CN subfolders: 22430
Number of MCI subfolders: 35
Total number of files across all MCI subfolders: 5742


# Pre-processing using GAN architecture method

In [15]:
# Define the input and output directories
input_dir = "../ADNI"  # Folder containing .nii or .nii.gz files
split_dir = "../ttv_split_ADNI"  # Folder where split files will be saved
classes = ["AD", "CN", "MCI"]  # Categories

# Create train and test directories
train_dir = os.path.join(split_dir, "train")
test_dir = os.path.join(split_dir, "test")

# Loop through all NIfTI files in the input directory
for cat in classes:
    input_dir_cat = os.path.join(input_dir, cat)  # Path to category folder
    train_cat_dir = os.path.join(train_dir, cat)
    test_cat_dir = os.path.join(test_dir, cat)

    # Create category subfolders for train/test
    os.makedirs(train_cat_dir, exist_ok=True)
    os.makedirs(test_cat_dir, exist_ok=True)

    for nii_file in os.listdir(input_dir_cat):
        split = np.random.rand()  # Generate random number for splitting

        if split <= 0.80:  # 80% to train
            current_dir = train_cat_dir
        else:  # 20% to test
            current_dir = test_cat_dir

        # Copy .nii file to the corresponding split directory
        src = os.path.join(input_dir_cat, nii_file)
        dst = os.path.join(current_dir, nii_file)

        shutil.copy2(src, dst)  # Use shutil.copy2 to copy single files

print("All files have been processed!")

All files have been processed!


# Convert nii images to png

In [20]:
import nibabel as nib
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import PIL.Image as Image
import pandas as pd
import os
from os import walk, path, makedirs

In [59]:
# Specify whether you're processing "ad" (Alzheimer's) or "nor" (Normal) category
# dtype = "AD"  # Change to "nor" when processing normal subjects
img_dir = "../ttv_split_ADNI_png/"  # Output directory for PNG images
nifti_dir = "../ttv_split_ADNI"   # Input directory containing .nii files (train/test)

# A sample filename structure in ADNI
# Example: ttv_split_ad/test/ad/005_S_0814/MPR__GradWarp__B1_Correction__N3__Scaled/2006-08-30_09_32_32.0/S18390/...
#          .../ADNI_005_S_0814_MR_MPR__GradWarp__B1_Correction__N3__Scaled_Br_20070923123111793_S18390_I74591.nii"

In [60]:
def create_directory(filename):
    """
    Create the directory structure in the output folder based on input NIfTI file paths.
    Returns:
        for_csv: The path for CSV recording.
        current_dir: The directory where the .png files will be saved.
    """
    dir_list = filename.split("/")

    name = dir_list[-1].split('.')[0]  # Extract the base filename (excluding .nii)
    dir_list = dir_list[2:-1]  # Remove the last part of the path to avoid repeating the filename

    # Set the base output directory
    current_dir = img_dir
    for next_dir in dir_list:
        current_dir = os.path.join(current_dir, next_dir)
        if not os.path.isdir(current_dir):
            os.makedirs(current_dir)  # Create the directory if it does not exist

    # Return both CSV path and the final output directory where PNG images will be stored
    return current_dir, os.path.join(current_dir, name)

In [61]:
def make_images(destination, data):
    """
    Convert the 3D volumetric data into multiple 2D axial slices and save them as PNG files.
    """
    images = []
    
    # Extract axial slices (2D slices along the first axis)
    for i in range(data.shape[0]):
        array = np.array(data[i, :, :])  # Get a single axial slice
        images.append(array)

    # Normalize the images
    norm_images = []
    for array in images:
        max_element = np.amax(array)
        if max_element > 0:
            array = (array / max_element) * 255.0  # Normalize to 0-255 range
        norm_images.append(array)

    # Save images as PNG
    for img, i in zip(norm_images, range(len(norm_images))):
        im = Image.fromarray(img)
        if im.mode != 'L':
            im = im.convert('L')  # Convert to grayscale (if not already)
        im.save(f"{destination}_{len(norm_images) - 1 - i}.png")  # Save with reversed order (bottom to top)

In [ ]:
# List to store file paths and dimensions for the CSV file
name_list = []
count_list = []

# Walk through the directory and process each NIfTI file
for root, _, files in walk(nifti_dir):
    for file in files:
        if file.endswith('.nii'):
            filename = os.path.join(root, file)
            
            try:
                img = nib.load(filename)  # Load NIfTI image
            except Exception as e:
                print(f"Error loading NIfTI file: {e}")
            
            data = img.get_fdata()  # Convert to numpy array (use get_fdata instead of get_data)

            # Create directories and generate images
            for_csv, destination = create_directory(filename)
            make_images(destination, data)

            # Record the directory and shape of the data
            name_list.append(for_csv)
            count_list.append(data.shape)

# Create a DataFrame to save the paths and dimensions into a CSV
df = pd.DataFrame(data={"dir_name": name_list, "file_count": count_list})
df.to_csv(f"./counter_all_ttv_{dtype}.csv", sep=',', index=False)

print("All files have been processed and converted to PNG!")

Error loading NIfTI file: Cannot work out file type of "../ttv_split_ADNI/test/MCI/I82705.nii"
Error loading NIfTI file: Cannot work out file type of "../ttv_split_ADNI/test/MCI/I81387.nii"
Error loading NIfTI file: Cannot work out file type of "../ttv_split_ADNI/test/MCI/I87556.nii"
Error loading NIfTI file: Cannot work out file type of "../ttv_split_ADNI/test/MCI/I109533.nii"
Error loading NIfTI file: Cannot work out file type of "../ttv_split_ADNI/test/MCI/I82171.nii"
Error loading NIfTI file: Cannot work out file type of "../ttv_split_ADNI/test/MCI/I66801.nii"
Error loading NIfTI file: Cannot work out file type of "../ttv_split_ADNI/test/MCI/I90614.nii"
Error loading NIfTI file: Cannot work out file type of "../ttv_split_ADNI/test/MCI/I66987.nii"
Error loading NIfTI file: Cannot work out file type of "../ttv_split_ADNI/test/MCI/I91703.nii"
Error loading NIfTI file: Cannot work out file type of "../ttv_split_ADNI/test/MCI/I35467.nii"
Error loading NIfTI file: Cannot work out file ty

# Select relevant sequences of MRI images only (relevant to AD)

In [17]:
# The effects of AD are visible mostly on the middle part of the head, with the lower and upper portions being irrelevant.
# With that in mind, we empirically chose some thresholds and only kept the relevant middle parts (from the ~45 percentile slice to the ~80 percentile slice).
# However, the original volumetric data, and thus the .png data we used, did not have the same dimensions
# i.e. the subjects' heads were not split in the same number of slices, and said slices did not have the same dimensions.
# So, if the visit had 192 slices, only slices 92-140 were kept, etc.

import shutil
from os import walk, path, makedirs

# Specify the data type: "ad" for Alzheimer's and "nor" for normal
dtype = "nor"  # Change to "ad" for Alzheimer's
png_dir = f"ttv_split_{dtype}_png/"  # Directory containing the .png files
seq_dir = f"ttv_split_{dtype}_png_seq/"  # Directory where the relevant slices will be copied

# Loop through the files in the png_dir
for root, _, files in walk(png_dir):
    # Define the split category (train, test, or valid)
    if "test" in root:
        split = f"test/{dtype}/"
    elif "train" in root:
        split = f"train/{dtype}/"
    elif "valid" in root:
        split = f"valid/{dtype}/"

    # Ensure the split directory exists
    makedirs(path.join(seq_dir, split), exist_ok=True)

    for file in files:
        # Extract the slice number from the filename
        file_lst = file.split("_")  # Split on the underscore to get the slice number
        sequence = file_lst[-1].split('.')[0]  # Extract the slice number part

        # Get the number of slices in this subject's dataset
        num_slices = len(files)

        # Apply the thresholds based on the number of slices
        if num_slices == 192:
            if 95 <= int(sequence) <= 140:  # Keep slices from 95 to 140 (for 192 slices)
                shutil.copy(path.join(root, file), path.join(seq_dir, split))
        elif num_slices == 240:
            if 100 <= int(sequence) <= 180:  # Keep slices from 100 to 180 (for 240 slices)
                shutil.copy(path.join(root, file), path.join(seq_dir, split))
        elif num_slices == 256:
            if 120 <= int(sequence) <= 200:  # Keep slices from 120 to 200 (for 256 slices)
                shutil.copy(path.join(root, file), path.join(seq_dir, split))

print("Relevant slices have been copied!")

Relevant slices have been copied!


# Resize images and save

In [18]:
# Resize all images underneath directory 'data' to (160, 192), through Lanczos resampling.

import numpy as np
from os import path, makedirs, walk
from PIL import Image

# Base directory for the original images and the resized images
base_dir = "data"
resized_base_dir = "data_resized"

# Walk through all files under the 'data' directory
for root, _, files in walk(base_dir):
    for file in files:
        # Open the image file
        img = Image.open(path.join(root, file))
        
        # Resize the image to (160, 192) using Lanczos resampling
        img = img.resize((160, 192), resample=Image.LANCZOS)

        # Determine the corresponding directory in 'data_resized'
        relative_path = path.relpath(root, base_dir)  # Get relative path from 'data'
        resized_dir = path.join(resized_base_dir, relative_path)  # Directory in 'data_resized'

        # Create the resized directory if it doesn't exist
        makedirs(resized_dir, exist_ok=True)

        # Save the resized image to the corresponding directory
        img.save(path.join(resized_dir, file))

print("All images have been resized and saved!")

All images have been resized and saved!
